In [2]:
import sys
import os
sys.path.append(os.path.abspath(".."))  # 如果 notebooks 在项目根目录下的 notebooks/ 下

In [3]:
import torch
from time_moe.models.modeling_time_moe import TimeMoeForPrediction, TimeMoeConfig

# tiny config: just to sanity-check forward + routing path
cfg = TimeMoeConfig(
    input_size=24,
    hidden_size=128,
    intermediate_size=512,
    num_hidden_layers=2,
    num_attention_heads=4,
    num_key_value_heads=4,
    num_experts=4,
    num_experts_per_tok=2,
    max_position_embeddings=512,
    use_dense=False,
    apply_aux_loss=True,
    horizon_lengths=[1],
    _attn_implementation="eager",

    # stage-conditioned routing ON
    enable_stage_routing=True,
    stage_routing_num_fourier=8,
    stage_routing_max_freq=10.0,
)

m = TimeMoeForPrediction(cfg)

# check layers have stage_router
cnt = 0
for layer in m.model.layers:
    if hasattr(layer.ffn_layer, "enable_stage_routing") and layer.ffn_layer.enable_stage_routing:
        cnt += 1
print("stage routing enabled layers:", cnt, "/", cfg.num_hidden_layers)

# forward sanity
x = torch.randn(8, 64, 24)          # [B,L,F]
y = torch.randn(8, 64, 24)          # labels aligned to horizon=1
mask = torch.ones(8, 64)

out = m(input_ids=x, labels=y, loss_masks=mask)
print("loss:", float(out.loss))


/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


stage routing enabled layers: 2 / 2
loss: 12.191237449645996
